# Task 6 · round 3: tune the regressor (plan_t6.md)

**Now:** public **8.64M**, rank 107; tier 4.0 needs **≤ 8.5M (−1.6%)**. Shipped:
`coread=True, coread_k=5, n_bag=5` → backtest A 7,110,402, B 7,096,961.
Public has run ~1.4× the validation gain last time, so about **−1.2% on A** should be enough.

The regressor settings (`learning_rate=0.03, max_iter=500, min_samples_leaf=40`, 31 leaves,
early stopping on) were never tuned. Features take most of each fit, so they are built
**once** per backtest and every setting is scored on the cached matrices.

Seed noise is ~0.3%, so every setting is scored as a **3-seed bag** (what ships), and only
gains clearly beyond that count.

In [1]:
%load_ext autoreload
%autoreload 2

import itertools
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

from assignment_1.modules.models import GapModel, GapRegressor, split_last
from assignment_1.modules.tasks.task6 import TASK6_GAP_PARAMS, TASK6_REGRESSOR_PARAMS
from assignment_1.modules.util import load_csv

DATA_DIR = "../data"  # notebook runs from notebooks/
train = load_csv("task_6_train.csv", DATA_DIR)
trA, valA = split_last(train)
trB, valB = split_last(trA)
SHIPPED = {"A": 7_110_402, "B": 7_096_961}

## T1. Cache features (slow: ~1 min per backtest)
Same construction as `GapRegressor.fit` / `.predict`: train on the history before each user's
last gap, predict the held-out gap.

In [2]:
def cache(tr: pd.DataFrame, val: pd.DataFrame) -> dict:
    m = GapRegressor(base=GapModel(**TASK6_GAP_PARAMS), **TASK6_REGRESSOR_PARAMS)
    hist_b, last_b = split_last(tr)
    return {"X": m.features(hist_b, last_b), "y": np.log1p(last_b["target"].to_numpy()),
            "Xv": m.features(tr, val), "yv": val["target"].to_numpy()}

t0 = time.time()
C = {"A": cache(trA, valA)}
print(f"A cached in {time.time() - t0:.0f}s: train {C['A']['X'].shape}, val {C['A']['Xv'].shape}")

A cached in 59s: train (55403, 51), val (55403, 51)


## Helper: MAE of a bag of seeds on cached features

In [3]:
BASE = dict(loss="absolute_error", learning_rate=0.03, max_iter=500, min_samples_leaf=40)

def bag_mae(params: dict, b: str = "A", seeds=(0, 1, 2)) -> float:
    c = C[b]
    logp = np.mean([HistGradientBoostingRegressor(**{**BASE, **params}, random_state=s)
                    .fit(c["X"], c["y"]).predict(c["Xv"]) for s in seeds], axis=0)
    return float(np.mean(np.abs(np.expm1(logp) - c["yv"])))

results = []

def score(name: str, params: dict, b: str = "A", seeds=(0, 1, 2)) -> pd.DataFrame:
    t0 = time.time()
    mae = bag_mae(params, b, seeds)
    results.append({"name": name, "backtest": b, **params, "mae": mae,
                    "Δ% vs shipped": 100 * (mae / SHIPPED[b] - 1), "sec": round(time.time() - t0)})
    return pd.DataFrame(results).sort_values(["backtest", "mae"]).round(3)

## T2. Reference and noise
`shipped settings, 3 seeds` should land near 7.11M (shipped uses 5 seeds). The second row
uses seeds 3–5: the difference between the two rows is the noise floor.

In [4]:
score("shipped settings", {})
score("shipped settings, seeds 3-5", {}, seeds=(3, 4, 5))

,name,backtest,mae,Δ% vs shipped,sec
0,shipped settings,A,7116635.430,0.088,7
1,"shipped settings, seeds 3-5",A,7119663.804,0.130,5


## T3. One setting at a time (around the shipped values)

In [5]:
for name, params in [
    ("lr 0.05", {"learning_rate": 0.05}),
    ("lr 0.1", {"learning_rate": 0.1}),
    ("lr 0.02, 1000 iters", {"learning_rate": 0.02, "max_iter": 1000}),
    ("leaves 15", {"max_leaf_nodes": 15}),
    ("leaves 63", {"max_leaf_nodes": 63}),
    ("leaves 127", {"max_leaf_nodes": 127}),
    ("min leaf 20", {"min_samples_leaf": 20}),
    ("min leaf 100", {"min_samples_leaf": 100}),
    ("min leaf 200", {"min_samples_leaf": 200}),
    ("l2 1", {"l2_regularization": 1.0}),
    ("max_features 0.5", {"max_features": 0.5}),
    ("no early stopping", {"early_stopping": False}),
]:
    table = score(name, params)
table

,name,backtest,mae,Δ% vs shipped,sec,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features,early_stopping
7,leaves 127,A,7095366.719,-0.211,13,NaN,NaN,127.0,NaN,NaN,NaN,NaN
6,leaves 63,A,7104296.383,-0.086,8,NaN,NaN,63.0,NaN,NaN,NaN,NaN
2,lr 0.05,A,7105759.885,-0.065,5,0.05,NaN,NaN,NaN,NaN,NaN,NaN
3,lr 0.1,A,7105787.401,-0.065,3,0.10,NaN,NaN,NaN,NaN,NaN,NaN
4,"lr 0.02, 1000 iters",A,7111577.061,0.017,9,0.02,1000.0,NaN,NaN,NaN,NaN,NaN
13,no early stopping,A,7111731.019,0.019,9,NaN,NaN,NaN,NaN,NaN,NaN,False
12,max_features 0.5,A,7112802.774,0.034,6,NaN,NaN,NaN,NaN,NaN,0.5,NaN
8,min leaf 20,A,7113597.699,0.045,6,NaN,NaN,NaN,20.0,NaN,NaN,NaN
0,shipped settings,A,7116635.430,0.088,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,l2 1,A,7118004.627,0.107,6,NaN,NaN,NaN,NaN,1.0,NaN,NaN


## T4. Combine the winners
Edit `GRID` with the values that helped in T3 (keep the shipped value in each list).

In [6]:
GRID = {
    "learning_rate": [0.03, 0.05],
    "max_leaf_nodes": [31, 63],
    "min_samples_leaf": [40, 100],
}
for values in itertools.product(*GRID.values()):
    p = dict(zip(GRID, values))
    table = score(" · ".join(f"{k}={v}" for k, v in p.items()), p)
table.head(10)

,name,backtest,mae,Δ% vs shipped,sec,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features,early_stopping
7,leaves 127,A,7095366.719,-0.211,13,NaN,NaN,127.0,NaN,NaN,NaN,NaN
21,learning_rate=0.05 · max_leaf_nodes=63 · min_s...,A,7098415.572,-0.169,5,0.05,NaN,63.0,100.0,NaN,NaN,NaN
17,learning_rate=0.03 · max_leaf_nodes=63 · min_s...,A,7102604.404,-0.110,7,0.03,NaN,63.0,100.0,NaN,NaN,NaN
6,leaves 63,A,7104296.383,-0.086,8,NaN,NaN,63.0,NaN,NaN,NaN,NaN
16,learning_rate=0.03 · max_leaf_nodes=63 · min_s...,A,7104296.383,-0.086,8,0.03,NaN,63.0,40.0,NaN,NaN,NaN
2,lr 0.05,A,7105759.885,-0.065,5,0.05,NaN,NaN,NaN,NaN,NaN,NaN
18,learning_rate=0.05 · max_leaf_nodes=31 · min_s...,A,7105759.885,-0.065,5,0.05,NaN,31.0,40.0,NaN,NaN,NaN
3,lr 0.1,A,7105787.401,-0.065,3,0.10,NaN,NaN,NaN,NaN,NaN,NaN
20,learning_rate=0.05 · max_leaf_nodes=63 · min_s...,A,7107391.203,-0.042,5,0.05,NaN,63.0,40.0,NaN,NaN,NaN
4,"lr 0.02, 1000 iters",A,7111577.061,0.017,9,0.02,1000.0,NaN,NaN,NaN,NaN,NaN


## T5. Confirm the best on backtest B (slow: caches B first)

In [7]:
BEST = {}   # <- the best params from T3 / T4
C["B"] = cache(trB, valB)
score("shipped settings", {}, "B")
score("best", BEST, "B")

,name,backtest,mae,Δ% vs shipped,sec,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features,early_stopping
7,leaves 127,A,7095366.719,-0.211,13,NaN,NaN,127.0,NaN,NaN,NaN,NaN
21,learning_rate=0.05 · max_leaf_nodes=63 · min_s...,A,7098415.572,-0.169,5,0.05,NaN,63.0,100.0,NaN,NaN,NaN
17,learning_rate=0.03 · max_leaf_nodes=63 · min_s...,A,7102604.404,-0.110,7,0.03,NaN,63.0,100.0,NaN,NaN,NaN
6,leaves 63,A,7104296.383,-0.086,8,NaN,NaN,63.0,NaN,NaN,NaN,NaN
16,learning_rate=0.03 · max_leaf_nodes=63 · min_s...,A,7104296.383,-0.086,8,0.03,NaN,63.0,40.0,NaN,NaN,NaN
2,lr 0.05,A,7105759.885,-0.065,5,0.05,NaN,NaN,NaN,NaN,NaN,NaN
18,learning_rate=0.05 · max_leaf_nodes=31 · min_s...,A,7105759.885,-0.065,5,0.05,NaN,31.0,40.0,NaN,NaN,NaN
3,lr 0.1,A,7105787.401,-0.065,3,0.10,NaN,NaN,NaN,NaN,NaN,NaN
20,learning_rate=0.05 · max_leaf_nodes=63 · min_s...,A,7107391.203,-0.042,5,0.05,NaN,63.0,40.0,NaN,NaN,NaN
4,"lr 0.02, 1000 iters",A,7111577.061,0.017,9,0.02,1000.0,NaN,NaN,NaN,NaN,NaN


## T6. Diagnostic: a constant scale on the predictions
MAE's best constant multiplier on backtest A. Only a hint: if it is far from 1 the model is
biased; public runs ~1.2× higher than validation, which this cannot see.

In [8]:
c = C["A"]
logp = np.mean([HistGradientBoostingRegressor(**{**BASE, **BEST}, random_state=s)
                .fit(c["X"], c["y"]).predict(c["Xv"]) for s in (0, 1, 2)], axis=0)
p = np.expm1(logp)
pd.Series({m: np.mean(np.abs(m * p - c["yv"])) for m in (0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2)}).round(0)

0.80    7573851.0
0.90    7274889.0
0.95    7175122.0
1.00    7116635.0
1.05    7094522.0
1.10    7104505.0
1.20    7197292.0
dtype: float64

## Decision
- Ship if `best` beats the shipped settings on A and B by clearly more than the T2 noise
  (aim ≤ −1% on A): in `tasks/task6.py`, set
  `TASK6_REGRESSOR_PARAMS["regressor"] = HistGradientBoostingRegressor(**{**BASE, **BEST}, random_state=0)`,
  then `make check TASKS=6`, `make changes TASKS=6`, upload.
- If early stopping is turned off, seeds no longer change the model (no subsampling), so
  `n_bag` > 1 just repeats the same fit: set `n_bag=1`, or keep `max_features < 1` for diversity.